## **Librerias**

In [3]:
import pandas as pd
import numpy as np
import joblib
from itertools import combinations
from collections import Counter

## **Preparación de datos**

In [15]:
dataset_raw = pd.read_csv('../Data/Datos_Test_Semestres_analisis.csv')
data = dataset_raw[['ID', 'Color Test de predominancia']]
data = data.iloc[:20]
data

,ID,Color Test de predominancia
0,S01001,Tipo B - Verde
1,S01002,Tipo C - Rojo
2,S01003,Tipo B - Verde
3,S01004,Tipo B - Verde
4,S01005,Tipo A - Azul
5,S01006,Tipo A - Azul
6,S01008,Tipo B - Verde
7,S01009,Tipo D - Naranja
8,S01010,Tipo B - Verde
9,S01011,Tipo A - Azul


## **Configuraciones**

In [16]:
TEAM_SIZE = 4
TOP_N = 10

TIPOS = [
    'Tipo A - Azul',
    'Tipo B - Verde',
    'Tipo C - Rojo',
    'Tipo D - Naranja'
]

## **Funciones**

In [17]:
def calcular_shannon(proporciones):

    proporciones = [p for p in proporciones if p > 0]

    if len(proporciones) == 0:
        return 0

    return -sum(p*np.log(p) for p in proporciones)

def calcular_balance(proporciones):

    proporciones = [p for p in proporciones if p > 0]

    if len(proporciones) <= 1:
        return 0

    return 1 - np.std(proporciones)

## **Modelo**

In [10]:
model = joblib.load("decision_tree_scode.pkl")

## **Ejecución algoritmo**

In [25]:
def recomendar_equipos(data, team_size=4, top_n=10):
    resultados = []

    for equipo in combinations(data.index, team_size):

        grupo = data.loc[list(equipo)]

        conteos = Counter(grupo["Color Test de predominancia"])

        total = team_size

        azul = conteos.get("Tipo A - Azul",0)
        verde = conteos.get("Tipo B - Verde",0)
        rojo = conteos.get("Tipo C - Rojo",0)
        naranja = conteos.get("Tipo D - Naranja",0)

        prop_azul = azul/total
        prop_verde = verde/total
        prop_rojo = rojo/total
        prop_naranja = naranja/total

        proporciones = [
            prop_azul,
            prop_verde,
            prop_rojo,
            prop_naranja
        ]

        diversidad = calcular_shannon(proporciones)

        balance = calcular_balance(proporciones)

        dominancia = max(proporciones)

        X = pd.DataFrame({

            "diversidad_shannon":[diversidad],
            "dominancia_equipo":[dominancia],
            "balance_equipo":[balance]

        })

        cohesion = model.predict(X)[0]

        resultados.append({

            "IDs": grupo["ID"].tolist(),

            

            "Indice Cohesion Predicho": cohesion

        })

    resultados = pd.DataFrame(resultados)
    resultados = resultados.sort_values(
    "Indice Cohesion Predicho",
    ascending=False
    ).reset_index(drop=True)

    top_equipos = resultados.head(top_n)
    
    return top_equipos.to_dict(orient="records")

## **Resultados**

In [26]:
recomendaciones = recomendar_equipos(data, team_size=TEAM_SIZE, top_n=TOP_N)
recomendaciones

[{'IDs': ['S01001', 'S01002', 'S01004', 'S01008'],
  'Indice Cohesion Predicho': 0.8729805801648807},
 {'IDs': ['S01001', 'S01002', 'S01003', 'S01022'],
  'Indice Cohesion Predicho': 0.8729805801648807},
 {'IDs': ['S01001', 'S01002', 'S01003', 'S01004'],
  'Indice Cohesion Predicho': 0.8729805801648807},
 {'IDs': ['S01001', 'S01002', 'S01004', 'S01016'],
  'Indice Cohesion Predicho': 0.8729805801648807},
 {'IDs': ['S01001', 'S01002', 'S01004', 'S01012'],
  'Indice Cohesion Predicho': 0.8729805801648807},
 {'IDs': ['S01001', 'S01002', 'S01003', 'S01008'],
  'Indice Cohesion Predicho': 0.8729805801648807},
 {'IDs': ['S01001', 'S01002', 'S01004', 'S01010'],
  'Indice Cohesion Predicho': 0.8729805801648807},
 {'IDs': ['S01001', 'S01002', 'S01003', 'S01010'],
  'Indice Cohesion Predicho': 0.8729805801648807},
 {'IDs': ['S01001', 'S01002', 'S01004', 'S01022'],
  'Indice Cohesion Predicho': 0.8729805801648807},
 {'IDs': ['S01001', 'S01002', 'S01003', 'S01012'],
  'Indice Cohesion Predicho': 0